# Installing Ultralytics package
-Ultralytics is a Python package, which gives you tools to train and use YOLO models.

In [ ]:
!pip install ultralytics

In [ ]:
!pip install -q -U kagglehub


# Dataset Reading

In [ ]:

import os
import yaml
import kagglehub

# Download the public Kaggle dataset
dataset_root = kagglehub.dataset_download(
    "febinnemmanuel/helmet-safety-yolov8-dataset"
)

print("Downloaded to:", dataset_root)

# Inspect the downloaded directory
print("Contents:", os.listdir(dataset_root))

# Find data.yaml anywhere inside the downloaded dataset
DATASET_PATH = None

for root, dirs, files in os.walk(dataset_root):
    if "data.yaml" in files:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "data.yaml was not found inside the downloaded dataset."
    )

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

print("DATASET_PATH:", DATASET_PATH)
print("YAML_PATH:", YAML_PATH)
print("Dataset contents:", os.listdir(DATASET_PATH))


In [ ]:
import os
import shutil
import kagglehub

# Get the correct downloaded path dynamically
downloaded_path = kagglehub.dataset_download("febinnemmanuel/helmet-safety-yolov8-dataset")
SOURCE_PATH = os.path.join(downloaded_path, "content", "helmet_yolo")
DATASET_PATH = "/content/helmet_yolo"

if not os.path.exists(DATASET_PATH):
    shutil.copytree(SOURCE_PATH, DATASET_PATH)

print("Dataset successfully copied to:", DATASET_PATH)
print("Contents:", os.listdir(DATASET_PATH))

In [ ]:
import yaml
import os

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

data["path"] = DATASET_PATH

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(open(YAML_PATH).read())

In [ ]:
for split in ["train", "val", "test"]:
    folder = os.path.join(DATASET_PATH, "images", split)
    print(split, "exists:", os.path.isdir(folder))

    if os.path.isdir(folder):
        print("Number of files:", len(os.listdir(folder)))

In [ ]:
import os
import yaml

print("Dataset path:", DATASET_PATH)
print("\nDataset YAML:")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

print(data)

for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATASET_PATH, "images", split)
    label_dir = os.path.join(DATASET_PATH, "labels", split)

    print(f"\n{split.upper()}")
    print("Images:", len(os.listdir(image_dir)) if os.path.isdir(image_dir) else "MISSING")
    print("Labels:", len(os.listdir(label_dir)) if os.path.isdir(label_dir) else "MISSING")

# Yolov8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

# Model Baseline Training

In [ ]:
results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640
)

In [ ]:
metrics = model.val()
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

### Model Evaluation on the Test Set

In [ ]:
import os
from ultralytics import YOLO

best_model_path = "/kaggle/working/runs/detect/train/weights/best.pt"
BASELINE_BEST = best_model_path
best_model = YOLO(best_model_path)

test_metrics = best_model.val(split='test')

print("\n--- Test Dataset Performance Metrics ---")
print(f"mAP@50: {test_metrics.box.map50:.4f}")
print(f"mAP@50-95: {test_metrics.box.map:.4f}")
print(f"Precision (mp): {test_metrics.box.mp:.4f}")
print(f"Recall (mr): {test_metrics.box.mr:.4f}")

# Fine-Tuning YOLOv8n with Augmentation
The dataset is fixed, so nothing here changes the data. Augmentation is applied **on the fly during training only**; val and test images are never augmented.

Plan: start from the baseline `best.pt` above, fine-tune with a lower learning rate and augmentation, then evaluate mAP / precision / recall and compare with the baseline on the test split.

In [ ]:
import os, pandas as pd
from ultralytics import YOLO
from IPython.display import display



## Fine-tuning with augmentation
- Starts from the baseline weights (set `START_WEIGHTS = "yolov8n.pt"` instead for a from-scratch comparison run).
- Lower learning rate (`lr0=0.002`) so fine-tuning refines the baseline instead of overwriting it.
- Horizontal flip is on (a helmet looks the same mirrored); vertical flip is off (riders are never upside down).
- `close_mosaic=10` turns mosaic off for the last 10 epochs so the model finishes on realistic images.

In [ ]:
START_WEIGHTS = BASELINE_BEST        # or "yolov8n.pt"

ft_model = YOLO(START_WEIGHTS)

ft_results = ft_model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    seed=0,
    name="helmet_finetune_aug",

    # optimiser: gentle fine-tuning
    optimizer="SGD",
    lr0=0.002,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=1,

    # colour augmentation (lighting / weather / camera variation)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # geometric augmentation
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,

    # composition augmentation
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=10,
)

FT_DIR  = str(ft_model.trainer.save_dir)
FT_BEST = os.path.join(FT_DIR, "weights", "best.pt")
print("Fine-tuned weights:", FT_BEST)

## Evaluation: mAP / Precision / Recall (overall and per class)

In [ ]:
def evaluate(weights, split="val", name="eval"):
    m = YOLO(weights)
    r = m.val(data=YAML_PATH, split=split, imgsz=640, plots=True, name=name, verbose=False)
    rows = [{"class": "ALL", "Precision": r.box.mp, "Recall": r.box.mr,
             "mAP50": r.box.map50, "mAP50-95": r.box.map}]
    for i, c in enumerate(r.box.ap_class_index):
        rows.append({"class": r.names[int(c)], "Precision": r.box.p[i], "Recall": r.box.r[i],
                     "mAP50": r.box.ap50[i], "mAP50-95": r.box.ap[i]})
    return pd.DataFrame(rows).round(4), r

ft_val_df,  ft_val_r  = evaluate(FT_BEST, "val",  name="ft_val")
ft_test_df, ft_test_r = evaluate(FT_BEST, "test", name="ft_test")

print("=== Fine-tuned model: VALIDATION ===")
display(ft_val_df)
print("=== Fine-tuned model: TEST (final result) ===")
display(ft_test_df)

## Baseline vs Fine-tuned (test split)

In [ ]:
base_test_df, _ = evaluate(BASELINE_BEST, "test", name="baseline_test")

cmp_df = pd.concat({"Baseline": base_test_df.set_index("class"),
                    "Fine-tuned + aug": ft_test_df.set_index("class")}, axis=1)
display(cmp_df)

# Change per metric for the overall row
delta = (ft_test_df.set_index("class").loc["ALL"] - base_test_df.set_index("class").loc["ALL"]).round(4)
print("Change (fine-tuned minus baseline), overall:")
print(delta.to_string())

## Plots (test split): confusion matrix, PR curve, F1 curve

In [ ]:
from IPython.display import Image
test_dir = str(ft_test_r.save_dir)
for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png"]:
    p = os.path.join(test_dir, f)
    if os.path.exists(p):
        print(f)
        display(Image(p, width=600))

## Save results (optional, needs Google Drive mounted)

In [ ]:
import shutil
shutil.copytree(FT_DIR, "/kaggle/working/helmet_finetune_aug_saved", dirs_exist_ok=True)
shutil.copy(FT_BEST, "/kaggle/working/helmet_finetuned_best.pt")
print("Saved to /kaggle/working")

In [ ]:
mild_model = YOLO(BASELINE_BEST)

mild_results = mild_model.train(
    data=YAML_PATH,
    epochs=40,
    imgsz=640,
    batch=16,
    patience=15,
    seed=0,
    name="helmet_finetune_mild",

    optimizer="SGD",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=1,

    # keep: lighting and mirror variation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    fliplr=0.5,
    flipud=0.0,

    # mild only: small shift and scale
    translate=0.05,
    scale=0.25,

    # off: these distorted the objects most
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    mixup=0.0,

    mosaic=1.0,
    close_mosaic=10,
)

MILD_DIR  = str(mild_model.trainer.save_dir)
MILD_BEST = os.path.join(MILD_DIR, "weights", "best.pt")

mild_test_df, _ = evaluate(MILD_BEST, "test", name="mild_test")

cmp3 = pd.concat({"Baseline": base_test_df.set_index("class"),
                  "Strong aug": ft_test_df.set_index("class"),
                  "Mild aug": mild_test_df.set_index("class")}, axis=1)
display(cmp3)

In [ ]:
adamw_model = YOLO(BASELINE_BEST)

adamw_results = adamw_model.train(
    data=YAML_PATH,
    epochs=40,
    imgsz=640,
    batch=16,
    patience=15,
    seed=0,
    name="helmet_finetune_adamw_rot5",

    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    warmup_epochs=1,

    # the only augmentation change from the Ultralytics defaults
    degrees=5.0,
)

ADAMW_DIR  = str(adamw_model.trainer.save_dir)
ADAMW_BEST = os.path.join(ADAMW_DIR, "weights", "best.pt")
print("Weights:", ADAMW_BEST)

In [ ]:
adamw_test_df, adamw_test_r = evaluate(ADAMW_BEST, "test", name="adamw_rot5_test")

cmp4 = pd.concat({"Baseline": base_test_df.set_index("class"),
                  "Strong aug": ft_test_df.set_index("class"),
                  "Mild aug": mild_test_df.set_index("class"),
                  "AdamW + rot5": adamw_test_df.set_index("class")}, axis=1)
display(cmp4)

delta = (adamw_test_df.set_index("class") - base_test_df.set_index("class")).round(4)
print("AdamW + rot5 minus baseline, per class:")
display(delta)

In [ ]:
import numpy as np

def pr_at_conf(weights, conf=0.25, split="test"):
    m = YOLO(weights)
    r = m.val(data=YAML_PATH, split=split, imgsz=640, plots=False, verbose=False)
    i = int(np.argmin(np.abs(r.box.px - conf)))
    rows = []
    for k, c in enumerate(r.box.ap_class_index):
        rows.append({"class": r.names[int(c)],
                     "Precision": r.box.p_curve[k][i],
                     "Recall": r.box.r_curve[k][i]})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = ["ALL", df["Precision"].mean(), df["Recall"].mean()]
    return df.set_index("class").round(4)

fixed = pd.concat({
    "Baseline":     pr_at_conf(BASELINE_BEST),
    "Strong aug":   pr_at_conf(FT_BEST),
    "Mild aug":     pr_at_conf(MILD_BEST),
    "AdamW + rot5": pr_at_conf(ADAMW_BEST),
}, axis=1)
display(fixed)

# setup

In [ ]:
import os, glob, random, yaml
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ultralytics import YOLO

# Model to inspect: the baseline weights (change this path to inspect another run)
try:
    BEST_MODEL = BASELINE_BEST
except NameError:
    BEST_MODEL = "/kaggle/working/runs/detect/train/weights/best.pt"
assert os.path.exists(BEST_MODEL), f"Weights not found: {BEST_MODEL}"
print("Model:", BEST_MODEL)

def split_images(split):
    """List image files of a split ('val' or 'test') using the dataset yaml."""
    cfg = yaml.safe_load(open(YAML_PATH))
    root = cfg.get("path", os.path.dirname(YAML_PATH))
    d = cfg[split]
    d = d if os.path.isabs(d) else os.path.join(root, d)
    files = []
    for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):
        files += glob.glob(os.path.join(d, f"**/*.{ext}"), recursive=True)
    return sorted(files)

def label_path(img):
    p = img.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)
    return os.path.splitext(p)[0] + ".txt"

test_imgs, val_imgs = split_images("test"), split_images("val")
print(f"{len(test_imgs)} test images, {len(val_imgs)} val images")

# One evaluation pass on the test split; its curves feed both parts below
model = YOLO(BEST_MODEL)
r = model.val(data=YAML_PATH, split="test", imgsz=640, plots=True,
              name="dashboard_test", verbose=False)
names = [r.names[int(c)] for c in r.box.ap_class_index]
print("Classes:", names)

## Threshold and IoU sliders

In [ ]:
def pr_at(conf):
    i = int(np.argmin(np.abs(r.box.px - conf)))
    return pd.DataFrame({"class": names,
                         "precision": [round(float(r.box.p_curve[k][i]), 3) for k in range(len(names))],
                         "recall":    [round(float(r.box.r_curve[k][i]), 3) for k in range(len(names))]}
                        ).set_index("class")

conf_s = widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05, description="Confidence",
                             continuous_update=False, layout=widgets.Layout(width="420px"))
iou_s  = widgets.FloatSlider(value=0.70, min=0.10, max=0.95, step=0.05, description="IoU (NMS)",
                             continuous_update=False, layout=widgets.Layout(width="420px"))
src_d  = widgets.Dropdown(options=[("test", "test"), ("val", "val")], value="test", description="Split")
img_d  = widgets.Dropdown(options=[(os.path.basename(p), p) for p in test_imgs[:300]],
                          description="Image", layout=widgets.Layout(width="520px"))
rand_b = widgets.Button(description="Random image", button_style="info")
out    = widgets.Output()

def refresh(*_):
    if not img_d.value:
        return
    res = model.predict(img_d.value, conf=conf_s.value, iou=iou_s.value, imgsz=640, verbose=False)[0]
    shown = res.plot()[:, :, ::-1]
    counts = pd.Series([res.names[int(c)] for c in res.boxes.cls]).value_counts() if len(res.boxes) else pd.Series(dtype=int)
    with out:
        clear_output(wait=True)
        plt.figure(figsize=(9, 6)); plt.imshow(shown); plt.axis("off")
        plt.title(f"conf={conf_s.value:.2f}  iou={iou_s.value:.2f}  detections={len(res.boxes)}")
        plt.show()
        print("Detections per class:", counts.to_dict() if len(counts) else "none")
        print(f"\nTest-set precision / recall at conf={conf_s.value:.2f}:")
        display(pr_at(conf_s.value))

def on_split(_):
    imgs = test_imgs if src_d.value == "test" else val_imgs
    img_d.options = [(os.path.basename(p), p) for p in imgs[:300]]

def on_random(_):
    imgs = test_imgs if src_d.value == "test" else val_imgs
    img_d.value = random.choice(imgs[:300])

conf_s.observe(refresh, "value"); iou_s.observe(refresh, "value")
img_d.observe(refresh, "value");  src_d.observe(on_split, "value")
rand_b.on_click(on_random)

display(widgets.VBox([widgets.HBox([src_d, rand_b]), img_d, conf_s, iou_s, out]))
refresh()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(20, 11))
colors = plt.cm.tab10.colors
px = r.box.px

# 1) Precision / Recall / F1 vs confidence (per class)
for k, n in enumerate(names):
    ax[0, 0].plot(px, r.box.r_curve[k],  color=colors[k], ls="-",  label=f"{n} recall")
    ax[0, 0].plot(px, r.box.p_curve[k],  color=colors[k], ls="--", label=f"{n} precision")
    ax[0, 0].plot(px, r.box.f1_curve[k], color=colors[k], ls=":",  label=f"{n} F1")
ax[0, 0].axvline(0.25, color="gray", lw=1)
ax[0, 0].set(title="Precision / recall / F1 vs confidence (grey = 0.25)", xlabel="confidence", ylabel="score", xlim=(0, 1), ylim=(0, 1.02))
ax[0, 0].legend(fontsize=7, ncol=2)

# 2) Precision vs recall trade-off (as confidence varies)
for k, n in enumerate(names):
    ax[0, 1].plot(r.box.r_curve[k], r.box.p_curve[k], color=colors[k], label=n)
ax[0, 1].set(title="Precision vs recall trade-off", xlabel="recall", ylabel="precision", xlim=(0, 1.02), ylim=(0, 1.02))
ax[0, 1].legend()

# 3) AP vs IoU threshold
ious = np.arange(0.5, 1.0, 0.05)
for k, n in enumerate(names):
    ax[0, 2].plot(ious, r.box.all_ap[k], marker="o", color=colors[k], label=n)
ax[0, 2].set(title="AP vs IoU threshold", xlabel="IoU threshold", ylabel="AP", ylim=(0, 1.02))
ax[0, 2].legend()

# 4) mAP per class
x = np.arange(len(names)); w = 0.38
ax[1, 0].bar(x - w/2, r.box.ap50, w, label="mAP50")
ax[1, 0].bar(x + w/2, r.box.ap,   w, label="mAP50-95")
for i in range(len(names)):
    ax[1, 0].text(i - w/2, r.box.ap50[i] + 0.01, f"{r.box.ap50[i]:.2f}", ha="center", fontsize=9)
    ax[1, 0].text(i + w/2, r.box.ap[i] + 0.01,   f"{r.box.ap[i]:.2f}",   ha="center", fontsize=9)
ax[1, 0].set_xticks(x); ax[1, 0].set_xticklabels(names)
ax[1, 0].set(title=f"mAP per class (overall mAP50={r.box.map50:.3f}, mAP50-95={r.box.map:.3f})", ylim=(0, 1.05))
ax[1, 0].legend()

# 5) Confusion matrix (columns normalised: share of each true class)
cm = r.confusion_matrix.matrix.astype(float)
cm = cm / np.maximum(cm.sum(0, keepdims=True), 1e-9)
labels = names + ["background"]
im = ax[1, 1].imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax[1, 1].set_xticks(range(len(labels))); ax[1, 1].set_xticklabels(labels, rotation=30)
ax[1, 1].set_yticks(range(len(labels))); ax[1, 1].set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax[1, 1].text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                      color="white" if cm[i, j] > 0.5 else "black")
ax[1, 1].set(title="Confusion matrix (normalised by true class)", xlabel="true", ylabel="predicted")

# 6) Training curves from the run that produced these weights
res_csv = os.path.join(os.path.dirname(os.path.dirname(BEST_MODEL)), "results.csv")
if os.path.exists(res_csv):
    h = pd.read_csv(res_csv); h.columns = [c.strip() for c in h.columns]
    for col, ls in [("metrics/mAP50(B)", "-"), ("metrics/mAP50-95(B)", "-"),
                    ("metrics/precision(B)", "--"), ("metrics/recall(B)", "--")]:
        if col in h: ax[1, 2].plot(h["epoch"], h[col], ls=ls, label=col.replace("metrics/", ""))
    ax2 = ax[1, 2].twinx()
    for col in ("train/box_loss", "val/box_loss"):
        if col in h: ax2.plot(h["epoch"], h[col], ls=":", color="red" if "val" in col else "gray", label=col)
    ax2.set_ylabel("box loss"); ax2.legend(loc="center right", fontsize=8)
    ax[1, 2].set(title="Training curves (validation metrics + box loss)", xlabel="epoch", ylim=(0, 1.02))
    ax[1, 2].legend(loc="lower right", fontsize=8)
else:
    ax[1, 2].text(0.5, 0.5, "results.csv not found\nfor this weights file", ha="center"); ax[1, 2].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
N, CONF, IOU, SPLIT = 4, 0.25, 0.70, "val"     # edit these, then re-run
pool = val_imgs if SPLIT == "val" else test_imgs
sample = random.sample(pool, min(N, len(pool)))

def draw_gt(img_path):
    img = cv2.imread(img_path); h, w = img.shape[:2]
    lp = label_path(img_path)
    if os.path.exists(lp):
        for line in open(lp):
            p = line.split()
            if len(p) < 5: continue
            c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:5])
            x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
            x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(img, r.names.get(c, str(c)), (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    return img[:, :, ::-1]

fig, ax = plt.subplots(len(sample), 2, figsize=(14, 5 * len(sample)), squeeze=False)
for i, p in enumerate(sample):
    pred = model.predict(p, conf=CONF, iou=IOU, imgsz=640, verbose=False)[0]
    ax[i, 0].imshow(draw_gt(p));              ax[i, 0].set_title(f"Ground truth: {os.path.basename(p)}", fontsize=9)
    ax[i, 1].imshow(pred.plot()[:, :, ::-1]); ax[i, 1].set_title(f"Prediction (conf={CONF}, iou={IOU})", fontsize=9)
    ax[i, 0].axis("off"); ax[i, 1].axis("off")
plt.tight_layout(); plt.show()

## Final model

**The final model is the baseline YOLOv8n (50 epochs, `runs/detect/train/weights/best.pt`).**

Three fine-tuning runs from the baseline weights (strong augmentation, mild augmentation, AdamW with 5° rotation) did not improve on it. At the same confidence (0.25) on the test split, Without Helmet recall fell from 0.776 (baseline) to 0.640, 0.657 and 0.663, a drop of about 11-14 points. Precision rose by only a few points, which is a poor trade for a violation detector, where a missed rider is the costly error.

**Note on selection:** the baseline was chosen by comparing the runs on the test split, so the reported test numbers are slightly optimistic.

**Deployment confidence:** chosen on the validation split, not the test split (see the next cell for the value and the resulting test numbers).

In [ ]:
import os, json, shutil
import numpy as np, pandas as pd
from ultralytics import YOLO

FINAL_MODEL   = BASELINE_BEST
TARGET_CLASS  = "Without Helmet"   # the violation class
TARGET_RECALL = 0.80               # edit: the recall you want on violations
MIN_CONF_NOTE = 0.10               # warn if the chosen confidence falls below this

final = YOLO(FINAL_MODEL)

# 1) Choose the confidence on the VALIDATION split
rv = final.val(data=YAML_PATH, split="val", imgsz=640, plots=False, verbose=False)
cls_names = [rv.names[int(c)] for c in rv.box.ap_class_index]
k = cls_names.index(TARGET_CLASS)
px, rec, prec, f1 = rv.box.px, rv.box.r_curve[k], rv.box.p_curve[k], rv.box.f1_curve[k]

meets = np.where(rec >= TARGET_RECALL)[0]
if len(meets):
    i = int(meets[-1])                 # highest confidence that still meets the target recall
    rule = f"highest confidence with {TARGET_CLASS} recall >= {TARGET_RECALL} on val"
else:
    i = int(np.argmax(f1))             # target not reachable: fall back to best F1
    rule = f"best F1 on val ({TARGET_CLASS} recall target {TARGET_RECALL} not reachable)"
DEPLOY_CONF = float(round(px[i], 2))
print(f"Deployment confidence = {DEPLOY_CONF}  ({rule})")
print(f"On val at this confidence: {TARGET_CLASS} recall={rec[i]:.3f}, precision={prec[i]:.3f}")
if DEPLOY_CONF < MIN_CONF_NOTE:
    print("Note: very low confidence, expect many false alarms. Check the sliders before using it.")

# 2) Report the final numbers on the TEST split at that confidence
rt = final.val(data=YAML_PATH, split="test", imgsz=640, plots=False, verbose=False)
j = int(np.argmin(np.abs(rt.box.px - DEPLOY_CONF)))
rows = []
for kk, c in enumerate(rt.box.ap_class_index):
    rows.append({"class": rt.names[int(c)],
                 "Precision": rt.box.p_curve[kk][j], "Recall": rt.box.r_curve[kk][j],
                 "mAP50": rt.box.ap50[kk], "mAP50-95": rt.box.ap[kk]})
final_df = pd.DataFrame(rows).set_index("class")
final_df.loc["ALL"] = [final_df["Precision"].mean(), final_df["Recall"].mean(), rt.box.map50, rt.box.map]
print(f"\nFINAL MODEL: test split, precision/recall at conf={DEPLOY_CONF} (mAP is threshold-independent)")
display(final_df.round(4))

# 3) Save the final weights and settings
os.makedirs("/kaggle/working", exist_ok=True)
shutil.copy(FINAL_MODEL, "/kaggle/working/final_helmet_model_baseline.pt")
with open("/kaggle/working/final_model_summary.json", "w") as f:
    json.dump({"weights": "final_helmet_model_baseline.pt", "deploy_conf": DEPLOY_CONF,
               "nms_iou": 0.7, "imgsz": 640, "selection_rule": rule,
               "test_metrics": final_df.round(4).reset_index().to_dict(orient="records")}, f, indent=2)
print("\nSaved final_helmet_model_baseline.pt and final_model_summary.json to /kaggle/working")
print(f"Use it with: model.predict(source=..., conf={DEPLOY_CONF}, iou=0.7, imgsz=640)")